# Introduction

Since Jan. 1, 2015, [The Washington Post](https://www.washingtonpost.com/) has been compiling a database of every fatal shooting in the US by a police officer in the line of duty. 

<center><img src=https://i.imgur.com/sX3K62b.png></center>

While there are many challenges regarding data collection and reporting, The Washington Post has been tracking more than a dozen details about each killing. This includes the race, age and gender of the deceased, whether the person was armed, and whether the victim was experiencing a mental-health crisis. The Washington Post has gathered this supplemental information from law enforcement websites, local new reports, social media, and by monitoring independent databases such as "Killed by police" and "Fatal Encounters". The Post has also conducted additional reporting in many cases.

There are 4 additional datasets: US census data on poverty rate, high school graduation rate, median household income, and racial demographics. [Source of census data](https://factfinder.census.gov/faces/nav/jsf/pages/community_facts.xhtml).

### Upgrade Plotly

Run the cell below if you are working with Google Colab

In [ ]:
%pip install --upgrade plotly

     |████████████████████████████████| 13.1MB 310kB/s 
  Found existing installation: plotly 4.4.1
    Uninstalling plotly-4.4.1:
      Successfully uninstalled plotly-4.4.1


## Import Statements

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# This might be helpful:
from collections import Counter

## Notebook Presentation

In [ ]:
pd.options.display.float_format = '{:,.2f}'.format

## Load the Data

In [ ]:
df_hh_income = pd.read_csv('Median_Household_Income_2015.csv', encoding="windows-1252")
df_pct_poverty = pd.read_csv('Pct_People_Below_Poverty_Level.csv', encoding="windows-1252")
df_pct_completed_hs = pd.read_csv('Pct_Over_25_Completed_High_School.csv', encoding="windows-1252")
df_share_race_city = pd.read_csv('Share_of_Race_By_City.csv', encoding="windows-1252")
df_fatalities = pd.read_csv('Deaths_by_Police_US.csv', encoding="windows-1252")

# Preliminary Data Exploration

* What is the shape of the DataFrames? 
* How many rows and columns do they have?
* What are the column names?
* Are there any NaN values or duplicates?

In [ ]:
# Explore the fatalities dataset
print('=== Deaths by Police ===')
print(f'Shape: {df_fatalities.shape}')
print(f'Columns: {df_fatalities.columns.tolist()}')
df_fatalities.head()

In [ ]:
# Explore the census datasets
print('=== Poverty Data ===')
print(f'Shape: {df_pct_poverty.shape}, Columns: {df_pct_poverty.columns.tolist()}')
print(f'\n=== High School Graduation Data ===')
print(f'Shape: {df_pct_completed_hs.shape}, Columns: {df_pct_completed_hs.columns.tolist()}')
print(f'\n=== Median Household Income Data ===')
print(f'Shape: {df_hh_income.shape}, Columns: {df_hh_income.columns.tolist()}')
print(f'\n=== Share of Race by City Data ===')
print(f'Shape: {df_share_race_city.shape}, Columns: {df_share_race_city.columns.tolist()}')

In [ ]:
# Check for NaN values and duplicates across all datasets
print('=== NaN Values ===')
for name, df in [('Fatalities', df_fatalities), ('Poverty', df_pct_poverty),
                  ('High School', df_pct_completed_hs), ('Income', df_hh_income),
                  ('Race', df_share_race_city)]:
    nans = df.isnull().sum().sum()
    dupes = df.duplicated().sum()
    print(f'{name}: {nans} NaN values, {dupes} duplicates')

print('\n=== Fatalities NaN per column ===')
print(df_fatalities.isnull().sum())

print('\n=== Fatalities Data Types ===')
print(df_fatalities.dtypes)

## Data Cleaning - Check for Missing Values and Duplicates

Consider how to deal with the NaN values. Perhaps substituting 0 is appropriate. 

In [ ]:
# Convert numeric columns in census data (they may contain '-' or other non-numeric values)
df_pct_poverty['poverty_rate'] = pd.to_numeric(df_pct_poverty['poverty_rate'], errors='coerce')
df_pct_completed_hs['percent_completed_hs'] = pd.to_numeric(df_pct_completed_hs['percent_completed_hs'], errors='coerce')
df_hh_income['Median Income'] = pd.to_numeric(df_hh_income['Median Income'], errors='coerce')

# Convert race share columns to numeric
race_cols = ['share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic']
for col in race_cols:
    df_share_race_city[col] = pd.to_numeric(df_share_race_city[col], errors='coerce')

print('Conversion complete. Checking NaN counts after conversion:')
print(f'Poverty NaN: {df_pct_poverty["poverty_rate"].isna().sum()}')
print(f'HS NaN: {df_pct_completed_hs["percent_completed_hs"].isna().sum()}')
print(f'Income NaN: {df_hh_income["Median Income"].isna().sum()}')

In [ ]:
# Fill NaN values with 0 for census data
df_pct_poverty['poverty_rate'] = df_pct_poverty['poverty_rate'].fillna(0)
df_pct_completed_hs['percent_completed_hs'] = df_pct_completed_hs['percent_completed_hs'].fillna(0)
df_hh_income['Median Income'] = df_hh_income['Median Income'].fillna(0)

for col in race_cols:
    df_share_race_city[col] = df_share_race_city[col].fillna(0)

# Convert date column in fatalities
df_fatalities['date'] = pd.to_datetime(df_fatalities['date'])

print('Data cleaning complete!')
print(f'\nFatalities date range: {df_fatalities["date"].min()} to {df_fatalities["date"].max()}')
df_fatalities.info()

# Chart the Poverty Rate in each US State

Create a bar chart that ranks the poverty rate from highest to lowest by US state. Which state has the highest poverty rate? Which state has the lowest poverty rate?  Bar Plot

In [ ]:
# Calculate average poverty rate by state
poverty_by_state = df_pct_poverty.groupby('Geographic Area')['poverty_rate'].mean().sort_values(ascending=False)
print(f'Highest poverty rate: {poverty_by_state.index[0]} ({poverty_by_state.iloc[0]:.2f}%)')
print(f'Lowest poverty rate: {poverty_by_state.index[-1]} ({poverty_by_state.iloc[-1]:.2f}%)')

In [ ]:
# Bar chart: poverty rate by state
fig = px.bar(
    x=poverty_by_state.index,
    y=poverty_by_state.values,
    title='Poverty Rate by US State (Highest to Lowest)',
    labels={'x': 'State', 'y': 'Poverty Rate (%)'},
    color=poverty_by_state.values,
    color_continuous_scale='Reds'
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

# Chart the High School Graduation Rate by US State

Show the High School Graduation Rate in ascending order of US States. Which state has the lowest high school graduation rate? Which state has the highest?

In [ ]:
# Average high school graduation rate by state (ascending)
hs_by_state = df_pct_completed_hs.groupby('Geographic Area')['percent_completed_hs'].mean().sort_values()

print(f'Lowest HS graduation rate: {hs_by_state.index[0]} ({hs_by_state.iloc[0]:.2f}%)')
print(f'Highest HS graduation rate: {hs_by_state.index[-1]} ({hs_by_state.iloc[-1]:.2f}%)')

fig = px.bar(
    x=hs_by_state.index,
    y=hs_by_state.values,
    title='High School Graduation Rate by US State (Ascending)',
    labels={'x': 'State', 'y': 'High School Graduation Rate (%)'},
    color=hs_by_state.values,
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

# Visualise the Relationship between Poverty Rates and High School Graduation Rates

#### Create a line chart with two y-axes to show if the rations of poverty and high school graduation move together.  

In [ ]:
# Prepare data: align poverty and HS rates by state
poverty_sorted = df_pct_poverty.groupby('Geographic Area')['poverty_rate'].mean().sort_values(ascending=False)
hs_sorted = df_pct_completed_hs.groupby('Geographic Area')['percent_completed_hs'].mean()
# Reindex HS to match poverty order
hs_aligned = hs_sorted.reindex(poverty_sorted.index)

In [ ]:
# Dual y-axis line chart: Poverty vs HS Graduation
fig, ax1 = plt.subplots(figsize=(16, 5))

ax1.set_xlabel('US State')
ax1.set_ylabel('Poverty Rate (%)', color='#e74c3c')
ax1.plot(poverty_sorted.index, poverty_sorted.values, color='#e74c3c', marker='o', markersize=3, label='Poverty Rate')
ax1.tick_params(axis='y', labelcolor='#e74c3c')
plt.xticks(rotation=90, fontsize=8)

ax2 = ax1.twinx()
ax2.set_ylabel('HS Graduation Rate (%)', color='#3498db')
ax2.plot(poverty_sorted.index, hs_aligned.values, color='#3498db', marker='s', markersize=3, label='HS Graduation Rate')
ax2.tick_params(axis='y', labelcolor='#3498db')

plt.title('Poverty Rate vs High School Graduation Rate by State', fontsize=14)
fig.tight_layout()
plt.show()

#### Now use a Seaborn .jointplot() with a Kernel Density Estimate (KDE) and/or scatter plot to visualise the same relationship

In [ ]:
# Merge poverty and HS data by state for scatter analysis
merged_state = pd.DataFrame({
    'Poverty Rate': poverty_sorted,
    'HS Graduation Rate': hs_sorted
}).dropna()

In [ ]:
# Seaborn jointplot with KDE
sns.set_style('whitegrid')
g = sns.jointplot(
    data=merged_state,
    x='Poverty Rate',
    y='HS Graduation Rate',
    kind='kde',
    fill=True,
    cmap='coolwarm'
)
g.fig.suptitle('Poverty Rate vs HS Graduation Rate (KDE)', y=1.02)
plt.show()

#### Seaborn's `.lmplot()` or `.regplot()` to show a linear regression between the poverty ratio and the high school graduation ratio. 

In [ ]:
# Seaborn regression plot
plt.figure(figsize=(10, 6))
sns.regplot(
    data=merged_state,
    x='Poverty Rate',
    y='HS Graduation Rate',
    scatter_kws={'alpha': 0.6, 's': 50},
    line_kws={'color': '#e74c3c'}
)
plt.title('Linear Regression: Poverty Rate vs HS Graduation Rate', fontsize=14)
plt.show()

# Create a Bar Chart with Subsections Showing the Racial Makeup of Each US State

Visualise the share of the white, black, hispanic, asian and native american population in each US State using a bar chart with sub sections. 

In [ ]:
# Average racial share by state
race_by_state = df_share_race_city.groupby('Geographic area')[race_cols].mean()
race_by_state.columns = ['White', 'Black', 'Native American', 'Asian', 'Hispanic']
race_by_state = race_by_state.sort_index()
race_by_state.head()

In [ ]:
# Stacked bar chart: racial makeup by state
fig = px.bar(
    race_by_state,
    x=race_by_state.index,
    y=['White', 'Black', 'Native American', 'Asian', 'Hispanic'],
    title='Racial Makeup of Each US State',
    labels={'value': 'Share (%)', 'variable': 'Race', 'Geographic area': 'State'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2,
    height=600
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Create Donut Chart by of People Killed by Race

Hint: Use `.value_counts()`

In [ ]:
# Value counts of race in fatalities
race_labels = {
    'W': 'White', 'B': 'Black', 'H': 'Hispanic',
    'A': 'Asian', 'N': 'Native American', 'O': 'Other'
}
race_counts = df_fatalities['race'].value_counts().dropna()
race_counts.index = race_counts.index.map(race_labels)
print(race_counts)

In [ ]:
# Donut chart: people killed by race
fig = px.pie(
    values=race_counts.values,
    names=race_counts.index,
    title='People Killed by Police — by Race',
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.show()

# Create a Chart Comparing the Total Number of Deaths of Men and Women

Use `df_fatalities` to illustrate how many more men are killed compared to women. 

In [ ]:
# Gender distribution
gender_counts = df_fatalities['gender'].value_counts()
print(gender_counts)
print(f'\nMen are killed {gender_counts["M"] / gender_counts["F"]:.1f}x more than women.')

In [ ]:
# Bar chart: deaths by gender
fig = px.bar(
    x=gender_counts.index.map({'M': 'Male', 'F': 'Female'}),
    y=gender_counts.values,
    title='Total Number of Deaths: Men vs Women',
    labels={'x': 'Gender', 'y': 'Number of Deaths'},
    color=gender_counts.index.map({'M': 'Male', 'F': 'Female'}),
    color_discrete_map={'Male': '#3498db', 'Female': '#e74c3c'}
)
fig.update_layout(showlegend=False)
fig.show()

# Create a Box Plot Showing the Age and Manner of Death

Break out the data by gender using `df_fatalities`. Is there a difference between men and women in the manner of death? 

In [ ]:
# Basic stats on age
print('Age statistics:')
print(df_fatalities['age'].describe())

In [ ]:
# Box plot: age by manner of death, split by gender
df_fatalities['gender_label'] = df_fatalities['gender'].map({'M': 'Male', 'F': 'Female'})

fig = px.box(
    df_fatalities.dropna(subset=['age']),
    x='manner_of_death',
    y='age',
    color='gender_label',
    title='Age and Manner of Death by Gender',
    labels={'manner_of_death': 'Manner of Death', 'age': 'Age', 'gender_label': 'Gender'},
    color_discrete_map={'Male': '#3498db', 'Female': '#e74c3c'}
)
fig.show()

In [ ]:
# Violin plot for a different view
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df_fatalities.dropna(subset=['age']),
    x='manner_of_death',
    y='age',
    hue='gender_label',
    split=True,
    palette={'Male': '#3498db', 'Female': '#e74c3c'}
)
plt.title('Age Distribution by Manner of Death and Gender')
plt.show()

# Were People Armed? 

In what percentage of police killings were people armed? Create chart that show what kind of weapon (if any) the deceased was carrying. How many of the people killed by police were armed with guns versus unarmed? 

In [ ]:
# Top weapons carried
armed_counts = df_fatalities['armed'].value_counts()
unarmed_count = armed_counts.get('unarmed', 0)
total = len(df_fatalities)
armed_pct = (1 - unarmed_count / total) * 100
print(f'People armed: {armed_pct:.1f}%')
print(f'People unarmed: {unarmed_count} ({100 - armed_pct:.1f}%)')
print(f'\nTop 10 weapons:')
print(armed_counts.head(10))

In [ ]:
# Bar chart: top 15 weapon types
top_weapons = armed_counts.head(15)

fig = px.bar(
    x=top_weapons.values,
    y=top_weapons.index,
    orientation='h',
    title='Weapons Carried by People Killed by Police (Top 15)',
    labels={'x': 'Count', 'y': 'Weapon Type'},
    color=top_weapons.values,
    color_continuous_scale='OrRd'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

In [ ]:
# Pie chart: armed vs unarmed
armed_simple = df_fatalities['armed'].apply(
    lambda x: 'Unarmed' if x == 'unarmed' else ('Gun' if x == 'gun' else 'Other Weapon')
).value_counts()

fig = px.pie(
    values=armed_simple.values,
    names=armed_simple.index,
    title='Armed Status: Gun vs Unarmed vs Other',
    color_discrete_sequence=['#e74c3c', '#f39c12', '#2ecc71'],
    hole=0.35
)
fig.update_traces(textinfo='percent+label+value')
fig.show()

# How Old Were the People Killed?

Work out what percentage of people killed were under 25 years old.  

In [ ]:
# Percentage of people killed under 25
under_25 = df_fatalities[df_fatalities['age'] < 25].shape[0]
total_with_age = df_fatalities['age'].dropna().shape[0]
pct_under_25 = (under_25 / total_with_age) * 100
print(f'People killed under 25: {under_25} out of {total_with_age}')
print(f'Percentage: {pct_under_25:.1f}%')

Create a histogram and KDE plot that shows the distribution of ages of the people killed by police. 

In [ ]:
# Histogram and KDE of age distribution
plt.figure(figsize=(12, 5))
sns.histplot(df_fatalities['age'].dropna(), bins=40, kde=True, color='#3498db', edgecolor='white')
plt.title('Age Distribution of People Killed by Police', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

Create a seperate KDE plot for each race. Is there a difference between the distributions? 

In [ ]:
# KDE plot per race
plt.figure(figsize=(12, 5))
race_map = {'W': 'White', 'B': 'Black', 'H': 'Hispanic', 'A': 'Asian', 'N': 'Native American', 'O': 'Other'}
for race_code, race_name in race_map.items():
    subset = df_fatalities[df_fatalities['race'] == race_code]['age'].dropna()
    if len(subset) > 10:
        sns.kdeplot(subset, label=race_name, fill=True, alpha=0.3)

plt.title('Age Distribution by Race (KDE)', fontsize=14)
plt.xlabel('Age')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()

# Race of People Killed

Create a chart that shows the total number of people killed by race. 

In [ ]:
# Total people killed by race
race_totals = df_fatalities['race'].value_counts().dropna()
race_totals.index = race_totals.index.map(race_labels)
print(race_totals)

In [ ]:
# Bar chart: people killed by race
fig = px.bar(
    x=race_totals.index,
    y=race_totals.values,
    title='Total Number of People Killed by Police — by Race',
    labels={'x': 'Race', 'y': 'Number of Deaths'},
    color=race_totals.index,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(showlegend=False)
fig.show()

# Mental Illness and Police Killings

What percentage of people killed by police have been diagnosed with a mental illness?

In [ ]:
# Mental illness statistics
mental_counts = df_fatalities['signs_of_mental_illness'].value_counts()
mental_pct = (mental_counts[True] / len(df_fatalities)) * 100
print(f'People with signs of mental illness: {mental_counts[True]} ({mental_pct:.1f}%)')
print(f'People without: {mental_counts[False]} ({100 - mental_pct:.1f}%)')

In [ ]:
# Pie chart: mental illness
fig = px.pie(
    values=mental_counts.values,
    names=['No Mental Illness', 'Signs of Mental Illness'],
    title='Police Killings: Signs of Mental Illness',
    color_discrete_sequence=['#3498db', '#e74c3c'],
    hole=0.4
)
fig.update_traces(textinfo='percent+label+value')
fig.show()

# In Which Cities Do the Most Police Killings Take Place?

Create a chart ranking the top 10 cities with the most police killings. Which cities are the most dangerous?  

In [ ]:
# Top 10 cities
top_cities = df_fatalities['city'].value_counts().head(10)
print('Top 10 most dangerous cities for police killings:')
print(top_cities)

In [ ]:
# Bar chart: top 10 cities
fig = px.bar(
    x=top_cities.values,
    y=top_cities.index,
    orientation='h',
    title='Top 10 Cities with Most Police Killings',
    labels={'x': 'Number of Killings', 'y': 'City'},
    color=top_cities.values,
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, showlegend=False)
fig.show()

# Rate of Death by Race

Find the share of each race in the top 10 cities. Contrast this with the top 10 cities of police killings to work out the rate at which people are killed by race for each city. 

In [ ]:
# Get racial breakdown of killings in top 10 cities
top_city_names = top_cities.index.tolist()
killings_top_cities = df_fatalities[df_fatalities['city'].isin(top_city_names)]

kill_race_city = killings_top_cities.groupby(['city', 'race']).size().unstack(fill_value=0)
kill_race_city.columns = kill_race_city.columns.map(lambda x: race_labels.get(x, x))
print('Killings by race in top 10 cities:')
kill_race_city

In [ ]:
# Stacked bar chart: killings by race in top 10 cities
fig = px.bar(
    kill_race_city,
    x=kill_race_city.index,
    y=kill_race_city.columns.tolist(),
    title='Police Killings by Race in Top 10 Cities',
    labels={'value': 'Number of Killings', 'variable': 'Race'},
    barmode='stack',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Create a Choropleth Map of Police Killings by US State

Which states are the most dangerous? Compare your map with your previous chart. Are these the same states with high degrees of poverty? 

In [ ]:
# Police killings by state
killings_by_state = df_fatalities['state'].value_counts().reset_index()
killings_by_state.columns = ['State', 'Killings']

fig = px.choropleth(
    killings_by_state,
    locations='State',
    locationmode='USA-states',
    color='Killings',
    scope='usa',
    title='Police Killings by US State',
    color_continuous_scale='Reds'
)
fig.update_layout(height=600)
fig.show()

In [ ]:
# Compare with poverty rate map
poverty_state = df_pct_poverty.groupby('Geographic Area')['poverty_rate'].mean().reset_index()
poverty_state.columns = ['State', 'Poverty Rate']

fig = px.choropleth(
    poverty_state,
    locations='State',
    locationmode='USA-states',
    color='Poverty Rate',
    scope='usa',
    title='Poverty Rate by US State (for comparison)',
    color_continuous_scale='Oranges'
)
fig.update_layout(height=600)
fig.show()

# Number of Police Killings Over Time

Analyse the Number of Police Killings over Time. Is there a trend in the data? 

In [ ]:
# Extract year and month
df_fatalities['year'] = df_fatalities['date'].dt.year
df_fatalities['month'] = df_fatalities['date'].dt.month
df_fatalities['year_month'] = df_fatalities['date'].dt.to_period('M')

# Killings per year
killings_per_year = df_fatalities['year'].value_counts().sort_index()
print('Killings per year:')
print(killings_per_year)

In [ ]:
# Bar chart: killings per year
fig = px.bar(
    x=killings_per_year.index,
    y=killings_per_year.values,
    title='Number of Police Killings per Year',
    labels={'x': 'Year', 'y': 'Number of Killings'},
    color=killings_per_year.values,
    color_continuous_scale='Reds'
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Monthly trend with rolling average
monthly = df_fatalities.groupby('year_month').size()
monthly.index = monthly.index.to_timestamp()
rolling = monthly.rolling(window=6).mean()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(monthly.index, monthly.values, alpha=0.4, label='Monthly Killings', color='#e74c3c')
ax.plot(rolling.index, rolling.values, linewidth=2.5, label='6-Month Rolling Average', color='#2c3e50')
ax.set_title('Police Killings Month-on-Month', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Number of Killings')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Killings by race over time
race_over_time = df_fatalities.groupby(['year', 'race']).size().reset_index(name='Killings')
race_over_time['race'] = race_over_time['race'].map(race_labels)
race_over_time = race_over_time.dropna(subset=['race'])

fig = px.line(
    race_over_time,
    x='year',
    y='Killings',
    color='race',
    title='Police Killings Over Time by Race',
    labels={'year': 'Year', 'race': 'Race'},
    markers=True,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig.show()

# Epilogue

Now that you have analysed the data yourself, read [The Washington Post's analysis here](https://www.washingtonpost.com/graphics/investigations/police-shootings-database/).

In [ ]:
# Summary
print('=== Key Findings ===')
print(f'Total police killings in dataset: {len(df_fatalities)}')
print(f'Date range: {df_fatalities["date"].min().strftime("%Y-%m-%d")} to {df_fatalities["date"].max().strftime("%Y-%m-%d")}')
print(f'\nMost common race: {race_totals.index[0]} ({race_totals.iloc[0]})')
print(f'Gender ratio (M/F): {gender_counts["M"]}:{gender_counts["F"]}')
print(f'Armed with gun: {armed_counts.get("gun", 0)} ({armed_counts.get("gun", 0)/len(df_fatalities)*100:.1f}%)')
print(f'Unarmed: {armed_counts.get("unarmed", 0)} ({armed_counts.get("unarmed", 0)/len(df_fatalities)*100:.1f}%)')
print(f'Signs of mental illness: {mental_pct:.1f}%')
print(f'Under 25 years old: {pct_under_25:.1f}%')
print(f'\nMost dangerous city: {top_cities.index[0]} ({top_cities.iloc[0]} killings)')
print(f'Most dangerous state: {killings_by_state.iloc[0]["State"]} ({killings_by_state.iloc[0]["Killings"]} killings)')